# RiskLab · USTA — Informe del Backend FastAPI

**Proyecto Integrador — Teoría del Riesgo**  
Universidad Santo Tomás · Bogotá

Este notebook documenta y prueba el backend FastAPI construido para el proyecto.  
Cada sección corresponde a un endpoint de la API y muestra la petición HTTP,  
la respuesta y una visualización o tabla interpretativa.

> **Requisito previo:** el servidor debe estar corriendo antes de ejecutar las celdas:
> ```bash
> cd backend
> uvicorn app.main:app --reload
> ```


## 0. Configuración e imports


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from scipy import stats as sp

BASE_URL = 'http://localhost:8000'
pd.set_option('display.float_format', '{:.4f}'.format)

def get(path, **params):
    r = requests.get(f'{BASE_URL}{path}', params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def post(path, body):
    r = requests.post(f'{BASE_URL}{path}', json=body, timeout=120)
    r.raise_for_status()
    return r.json()

print('Setup listo. Base URL:', BASE_URL)


---
## 1. Estado de la API — `GET /health`

Verifica que el servidor FastAPI está activo.


In [ ]:
data = get('/health')
print(f"Estado  : {data['status']}")
print(f"Version : {data['version']}")
print(f"Timestamp: {data['timestamp']}")


---
## 2. Portafolio disponible — `GET /activos`

Lista los cinco activos del portafolio con su sector y color asignado.


In [ ]:
data = get('/activos')
print(f"Benchmark: {data['benchmark']} | Total: {data['total']} activos\n")
df = pd.DataFrame(data['activos'])
display(df[['ticker', 'nombre', 'sector']].style.set_caption('Activos del portafolio').hide(axis='index'))


---
## 3. Precios históricos — `GET /precios/{ticker}`

Descarga precios diarios de cierre ajustado.  
Se normalizan a base 100 para comparar rendimientos acumulados.


In [ ]:
tickers = ['AAPL', 'JPM', 'XOM', 'JNJ', 'AMZN']
colores = ['#A89060', '#3D8B6E', '#3A6B8A', '#5A4E7A', '#8B4A4A']

fig, ax = plt.subplots(figsize=(13, 5))
for tk, col in zip(tickers, colores):
    data = get(f'/precios/{tk}', years=3)
    fechas = pd.to_datetime([p['fecha'] for p in data['datos']])
    precios = [p['precio'] for p in data['datos']]
    base = precios[0]
    ax.plot(fechas, [p / base * 100 for p in precios], label=tk, color=col, lw=1.6)

ax.axhline(100, color='gray', ls='--', lw=0.8, alpha=0.5)
ax.set_title('Rendimiento acumulado — base 100 (3 años)', fontsize=13, fontweight='bold')
ax.set_ylabel('Indice (base 100)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Observaciones: {data['n_observaciones']} dias")


---
## 4. Rendimientos y propiedades empíricas — `GET /rendimientos/{ticker}`

Calcula log-rendimientos y estadísticos descriptivos completos.  
Incluye pruebas de normalidad **Jarque-Bera** y **Shapiro-Wilk**.


In [ ]:
tickers = ['AAPL', 'JPM', 'XOM', 'JNJ', 'AMZN']
rows = []
for tk in tickers:
    d = get(f'/rendimientos/{tk}')['estadisticos']
    rows.append({
        'Ticker':    tk,
        'Ret. anual':  f"{d['media_anual']*100:.2f}%",
        'Vol. anual':  f"{d['volatilidad_anual']*100:.2f}%",
        'Asimetria':   f"{d['asimetria']:.3f}",
        'Curtosis ex': f"{d['curtosis_exceso']:.3f}",
        'JB p-val':    f"{d['jarque_bera_pvalue']:.4f}",
        'Normal?':     'Si' if d['es_normal_jb'] else 'No',
    })

df = pd.DataFrame(rows).set_index('Ticker')
display(df.style.set_caption('Estadisticos de log-rendimientos diarios'))
print('\nNota: "No" = se rechaza normalidad (p < 0.05) — tipico en activos financieros (colas pesadas).')


In [ ]:
data_aapl = get('/rendimientos/AAPL')
rets = [p['rendimiento_log'] for p in data_aapl['datos']]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(rets, bins=60, density=True, color='#A89060', alpha=0.75, label='AAPL')
x = np.linspace(min(rets), max(rets), 300)
mu, sigma = np.mean(rets), np.std(rets)
ax.plot(x, sp.norm.pdf(x, mu, sigma), 'k--', lw=1.8, label='Normal teorica')
ax.set_title('Distribucion de log-retornos - AAPL')
ax.legend()
ax.grid(alpha=0.3)

ax2 = axes[1]
sp.probplot(rets, dist='norm', plot=ax2)
ax2.set_title('Q-Q Plot - AAPL vs Normal')
ax2.get_lines()[0].set(color='#A89060', markersize=2)
ax2.get_lines()[1].set(color='black', lw=1.5)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


---
## 5. Indicadores técnicos — `GET /indicadores/{ticker}`

El endpoint retorna siete series de indicadores.  
Se visualizan **RSI** y **MACD** para AAPL como ejemplo.


In [ ]:
data = get('/indicadores/AAPL', years=2)
N = 252
rsi_data  = [p for p in data['rsi']  if p['rsi']  is not None][-N:]
macd_data = [p for p in data['macd'] if p['macd'] is not None][-N:]

fechas_rsi  = pd.to_datetime([p['fecha'] for p in rsi_data])
vals_rsi    = [p['rsi'] for p in rsi_data]
fechas_macd = pd.to_datetime([p['fecha'] for p in macd_data])
vals_macd   = [p['macd'] for p in macd_data]
vals_sig    = [p['senal'] if 'senal' in p else p.get('señal', 0) for p in macd_data]
vals_hist   = [p['histograma'] for p in macd_data]

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

ax = axes[0]
ax.plot(fechas_rsi, vals_rsi, color='#A89060', lw=1.5, label='RSI(14)')
ax.axhline(70, color='#dc2626', lw=1, ls='--', alpha=0.7, label='Sobrecompra 70')
ax.axhline(30, color='#16a34a', lw=1, ls='--', alpha=0.7, label='Sobreventa 30')
ax.fill_between(fechas_rsi, vals_rsi, 70, where=[v > 70 for v in vals_rsi], alpha=0.15, color='#dc2626')
ax.fill_between(fechas_rsi, vals_rsi, 30, where=[v < 30 for v in vals_rsi], alpha=0.15, color='#16a34a')
ax.set_ylabel('RSI')
ax.set_ylim(0, 100)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_title('RSI(14) - AAPL (ultimo ano)')

ax2 = axes[1]
colors_hist = ['#16a34a' if v >= 0 else '#dc2626' for v in vals_hist]
ax2.bar(fechas_macd, vals_hist, color=colors_hist, alpha=0.5, label='Histograma')
ax2.plot(fechas_macd, vals_macd, color='#3A6B8A', lw=1.5, label='MACD')
ax2.plot(fechas_macd, vals_sig,  color='#8B4A4A', lw=1.2, ls='--', label='Senal')
ax2.axhline(0, color='gray', lw=0.8)
ax2.set_ylabel('MACD')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_title('MACD(12,26,9) - AAPL (ultimo ano)')

plt.tight_layout()
plt.show()


---
## 6. Valor en Riesgo y CVaR — `POST /var`

Calcula VaR con tres métodos y CVaR para el portafolio equi-ponderado (20% cada activo).  
Incluye **test de Kupiec** para backtesting del modelo histórico.


In [ ]:
payload = {
    'tickers':    ['AAPL', 'JPM', 'XOM', 'JNJ', 'AMZN'],
    'weights':    [0.20, 0.20, 0.20, 0.20, 0.20],
    'confidence': 0.95,
    'capital':    100_000,
    'years':      3,
}
data = post('/var', payload)

rows = []
for r in data['resultados']:
    rows.append({
        'Metodo':           r['metodo'],
        'VaR diario (%)':   f"{r['var_diario_pct']*100:.3f}%",
        'VaR diario (USD)': f"${r['var_diario_usd']:,.0f}",
        'VaR anual (%)':    f"{r['var_anual_pct']*100:.2f}%",
        'CVaR diario (%)':  f"{r['cvar_diario_pct']*100:.3f}%",
        'CVaR (USD)':       f"${r['cvar_diario_usd']:,.0f}",
    })

df = pd.DataFrame(rows).set_index('Metodo')
display(df.style.set_caption(f"VaR y CVaR al {data['confianza']*100:.0f}% - Capital ${data['capital']:,.0f}"))

k = data['kupiec']
print(f"\nTest de Kupiec (backtesting histerico):")
print(f"  Violaciones observadas : {k['n_violaciones']} ({k['tasa_violacion']*100:.2f}%)")
print(f"  Tasa esperada          : {k['tasa_esperada']*100:.2f}%")
print(f"  LR statistic / p-value : {k['lr_statistic']:.4f} / {k['p_value']:.4f}")
print(f"  Resultado              : {k['resultado']}")


In [ ]:
# Distribucion de retornos del portafolio con lineas VaR/CVaR
import requests, numpy as np, matplotlib.pyplot as plt

data_r = get('/rendimientos/AAPL')  # proxy de retornos simples
rets   = np.array([p['rendimiento_log'] for p in data_r['datos']])

# Recalcular VaR hist para graficar
var_h  = float(-np.percentile(rets, 5))
cvar_h = float(-rets[rets < -var_h].mean())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rets, bins=80, density=True, color='#3A6B8A', alpha=0.6, label='Retornos AAPL')
ax.axvline(-var_h,  color='#dc2626', lw=2, ls='--', label=f'VaR 95% = {var_h*100:.2f}%')
ax.axvline(-cvar_h, color='#7c1d1d', lw=2, ls=':',  label=f'CVaR 95% = {cvar_h*100:.2f}%')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 15],
                 min(rets), -var_h, alpha=0.12, color='#dc2626')
ax.set_title('Distribucion de retornos con VaR y CVaR')
ax.set_xlabel('Retorno diario')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 7. CAPM y riesgo sistematico — `GET /capm`

Estima **Beta**, **Alpha de Jensen** y rendimiento esperado CAPM.  
La tasa libre de riesgo (Rf) se obtiene automáticamente desde **^IRX** (T-Bill 3M).


In [ ]:
data = get('/capm', years=3)
print(f"Rf anual (^IRX) : {data['tasa_libre_riesgo']*100:.2f}%")
print(f"Rm anual (S&P500): {data['retorno_mercado']*100:.2f}%")
print(f"Prima de mercado: {data['prima_mercado']*100:.2f}%\n")

rows = []
for a in data['activos']:
    rows.append({
        'Ticker':       a['ticker'],
        'Beta':         f"{a['beta']:.4f}",
        'IC 95%':       f"[{a['beta_ic_inferior']:.3f}, {a['beta_ic_superior']:.3f}]",
        'Alpha Jensen': f"{a['alpha_jensen']*100:.2f}%",
        'R2':           f"{a['r_cuadrado']:.3f}",
        'E[R] CAPM':    f"{a['retorno_esperado_capm']*100:.2f}%",
        'Clasif.':      a['clasificacion'],
    })

df = pd.DataFrame(rows).set_index('Ticker')
display(df.style.set_caption('Resultados CAPM por activo'))


In [ ]:
# Security Market Line
rf = data['tasa_libre_riesgo']
rm = data['retorno_mercado']
colores = {'AAPL':'#A89060','JPM':'#3D8B6E','XOM':'#3A6B8A','JNJ':'#5A4E7A','AMZN':'#8B4A4A'}

betas = [a['beta'] for a in data['activos']]
b_line = np.linspace(min(betas) - 0.2, max(betas) + 0.2, 100)
sml = rf + b_line * (rm - rf)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(b_line, sml * 100, 'k--', lw=1.5, label='SML')
for a in data['activos']:
    ax.scatter(a['beta'], a['retorno_esperado_capm'] * 100,
               color=colores.get(a['ticker'], '#555'), s=110, zorder=5)
    ax.annotate(a['ticker'], (a['beta'], a['retorno_esperado_capm'] * 100),
                textcoords='offset points', xytext=(7, 4), fontsize=9)

ax.axvline(1, color='gray', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel('Beta')
ax.set_ylabel('Retorno esperado CAPM (%)')
ax.set_title('Security Market Line')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


---
## 8. Frontera eficiente — `POST /frontera-eficiente`

Simula 10 000 portafolios y encuentra los óptimos de Markowitz:  
**Mínima Varianza** y **Máximo Sharpe**.


In [ ]:
payload = {
    'tickers': ['AAPL', 'JPM', 'XOM', 'JNJ', 'AMZN'],
    'n_portfolios': 10000,
    'years': 3,
}
data = post('/frontera-eficiente', payload)
print(f"Portafolios simulados: {data['n_simulados']}")

vols    = [p['volatilidad'] for p in data['portafolios']]
rets    = [p['retorno']     for p in data['portafolios']]
sharpes = [p['sharpe']      for p in data['portafolios']]

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(vols, rets, c=sharpes, cmap='RdYlGn', alpha=0.35, s=4)
plt.colorbar(sc, ax=ax, label='Ratio de Sharpe')

estilos = {'Minima Varianza': ('o', '#3A6B8A', 140), 'Maximo Sharpe': ('*', '#8B4A4A', 220)}
for opt in data['optimos']:
    lbl = opt['nombre'].replace('í','i').replace('á','a').replace('é','e')
    key = next((k for k in estilos if k in lbl or lbl in k), list(estilos.keys())[0])
    m, c, sz = estilos.get(key, ('D', 'gray', 120))
    ax.scatter(opt['volatilidad'], opt['retorno'], marker=m, color=c, s=sz, zorder=10,
               label=f"{opt['nombre']} (Sharpe={opt['sharpe']:.2f})")
    ax.annotate(opt['nombre'], (opt['volatilidad'], opt['retorno']),
                textcoords='offset points', xytext=(8, -14), fontsize=8,
                color=c, fontweight='bold')

ax.set_xlabel('Volatilidad anual')
ax.set_ylabel('Retorno esperado anual')
ax.set_title('Frontera eficiente de Markowitz — 10 000 portafolios')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Composicion de portafolios optimos
for opt in data['optimos']:
    print(f"\n{opt['nombre']}")
    print(f"  Retorno: {opt['retorno']*100:.2f}%  Volatilidad: {opt['volatilidad']*100:.2f}%  Sharpe: {opt['sharpe']:.3f}")
    comp = dict(sorted(opt['composicion'].items(), key=lambda x: x[1], reverse=True))
    for tk, w in comp.items():
        bar = 'x' * int(w * 40)
        print(f"  {tk:5s}  {w*100:5.1f}%  {bar}")


---
## 9. Senales y alertas — `GET /alertas`

Evalúa MACD · RSI · Bollinger · Cruce de Medias · Estocástico  
y genera un score compuesto por activo.


In [ ]:
data = get('/alertas')
print(f"Timestamp: {data['timestamp'][:19]}\n")

print(f"{'Ticker':<8} {'Clasificacion':<22} {'Score':>7}")
print('-' * 42)
for a in data['activos']:
    print(f"{a['ticker']:<8} {a['clasificacion']:<22} {a['score_compuesto']:>7.3f}")

print("\nDetalle del primer activo:")
a0 = data['activos'][0]
df = pd.DataFrame(a0['senales'] if 'senales' in a0 else a0.get('señales', []))
if not df.empty:
    display(df[['indicador', 'señal', 'valor_actual', 'descripcion']]
            .style.set_caption(f"Senales activas: {a0['clasificacion']}").hide(axis='index'))


---
## 10. Contexto macroeconomico — `GET /macro`

Indicadores en tiempo real: Rf · Tesoro 10Y · VIX · USD/COP · EUR/USD · Oro.


In [ ]:
data = get('/macro')
print(f"Tasa libre de riesgo (^IRX): {data['tasa_libre_riesgo']*100:.2f}%")
print(f"Timestamp: {data['timestamp'][:19]}\n")

rows = []
for ind in data['indicadores']:
    rows.append({
        'Indicador': ind['nombre'],
        'Valor':     ind['valor'],
        'Unidad':    ind['unidad'],
        'Descripcion': ind['descripcion'],
    })
df = pd.DataFrame(rows)
display(df.style.set_caption('Indicadores macroeconomicos — tiempo real').hide(axis='index'))


---
## 11. Modelos ARCH/GARCH — `POST /garch`

Ajusta y compara cuatro especificaciones:  
**ARCH(1)** · **GARCH(1,1)** · **GJR-GARCH(1,1)** · **EGARCH(1,1)**  
El mejor se selecciona por **AIC minimo**.


In [ ]:
data = post('/garch', {'ticker': 'AAPL', 'years': 3})
print(f"Ticker       : {data['ticker']}")
print(f"Mejor modelo : {data['mejor_modelo']} (AIC minimo)")
print(f"JB residuos  : stat={data['jarque_bera_residuos']:.4f}  p={data['jarque_bera_pvalue']:.4f}\n")

rows = []
for s in data['especificaciones']:
    rows.append({
        'Modelo':      s['nombre'],
        'Log-Lik':     f"{s['log_likelihood']:.2f}",
        'AIC':         f"{s['aic']:.2f}",
        'BIC':         f"{s['bic']:.2f}",
        'Alpha':       f"{s['alpha']:.6f}",
        'Beta':        f"{s['beta']:.6f}" if s['beta'] is not None else 'N/A',
        'Persistencia':f"{s['persistencia']:.4f}" if s['persistencia'] is not None else 'N/A',
        'Vol anual':   f"{s['pronostico_vol_anual']*100:.2f}%",
    })

df = pd.DataFrame(rows).set_index('Modelo')
display(df.style.set_caption('Comparacion de modelos ARCH/GARCH — AAPL')
           .highlight_min(subset=['AIC', 'BIC'], color='#d4edda'))


In [ ]:
# Grafico comparativo AIC/BIC
nombres = [r['nombre'] for r in data['especificaciones']]
aics    = [r['aic']    for r in data['especificaciones']]
bics    = [r['bic']    for r in data['especificaciones']]

x = np.arange(len(nombres))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, aics, w, label='AIC', color='#3A6B8A', alpha=0.85)
ax.bar(x + w/2, bics, w, label='BIC', color='#8B4A4A', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(nombres, fontsize=9)
ax.set_title('Comparacion AIC/BIC — modelos GARCH · AAPL')
ax.set_ylabel('Criterio de informacion')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Menor AIC: {data['mejor_modelo']}")


---
## 12. Arquitectura tecnica del backend

### Estructura de archivos
```
backend/
├── app/
│   ├── config.py        ← BaseSettings + lru_cache  (variables de entorno)
│   ├── models.py        ← Modelos Pydantic v2        (request y response)
│   ├── services.py      ← Clases de logica           (DataService, RiskCalculator...)
│   ├── dependencies.py  ← Depends()                  (inyeccion de dependencias)
│   └── main.py          ← FastAPI app                (10 endpoints)
├── requirements.txt
└── .env.example
```

### Patrones implementados

| Patron | Implementacion |
|--------|----------------|
| **BaseSettings + .env** | `config.py` — API keys y parametros nunca hardcodeados |
| **Pydantic v2 validators** | `@field_validator` tickers · `@model_validator` pesos VaR |
| **Dependency Injection** | `Depends()` en todos los endpoints |
| **async / await** | Rutas async + `run_sync()` ejecuta yfinance en ThreadPool |
| **Decorador personalizado** | `@timed` en services.py — registra tiempo de ejecucion |
| **POO** | 7 clases: DataService, TechnicalIndicators, RiskCalculator, CAPMCalculator, PortfolioAnalyzer, SignalGenerator, MacroService |
| **Cache interno** | TTL 30 min en `_cache_get/set` — evita llamadas repetidas |
| **HTTP error codes** | 404 ticker no existe · 422 validacion · 503 API externa caida |

### Resumen de endpoints

| Metodo | Endpoint | Modulo del curso |
|--------|----------|------------------|
| GET | `/activos` | Portafolio |
| GET | `/precios/{ticker}` | Modulo 1 |
| GET | `/rendimientos/{ticker}` | Modulo 2 |
| GET | `/indicadores/{ticker}` | Modulo 1 |
| POST | `/var` | Modulo 5 |
| GET | `/capm` | Modulo 4 |
| POST | `/frontera-eficiente` | Modulo 6 |
| GET | `/alertas` | Modulo 7 |
| GET | `/macro` | Modulo 8 |
| POST | `/garch` | Modulo 3 |


---
## 13. Conclusiones

- **Separacion backend/frontend**: FastAPI actua como motor de calculo; Streamlit  
  consume los endpoints via HTTP siguiendo el patron cliente-servidor.

- **Datos en tiempo real**: precios, indicadores macro y tasa libre de riesgo se obtienen  
  dinamicamente desde Yahoo Finance con cache de 30 minutos.

- **Validacion robusta**: Pydantic garantiza que los pesos sumen 1.0, los tickers sean  
  validos y los parametros esten en rangos razonables antes de ejecutar cualquier calculo.

- **Senales automatizadas**: cinco indicadores generan un score compuesto que puede  
  guiar decisiones de inversion con umbrales configurables.

- **Modelos GARCH**: la comparacion AIC/BIC permite seleccionar la especificacion  
  optima de volatilidad condicional para cada activo.

- **Documentacion automatica**: Swagger UI en `/docs` describe todos los endpoints  
  con sus esquemas Pydantic sin documentacion adicional.
